In [14]:
import httpx
import time
import pandas as pd
import re
from transformers import pipeline
from datetime import datetime, timezone

In [15]:
## Functions I might need later:

# def dedupe(xs):
#     return list(dict.fromkeys([x for x in xs if x]))

# env_post_ids = dedupe(env_post_ids)
# infr_post_ids = dedupe(infr_post_ids)
# housing_post_ids = dedupe(housing_post_ids)
# econ_post_ids = dedupe(econ_post_ids)
# life_qual_post_ids = dedupe(life_qual_post_ids)
# aesth_post_ids = dedupe(aesth_post_ids)
# gov_post_ids = dedupe(gov_post_ids)
# not_useful_post_ids = dedupe(not_useful_post_ids)

In [18]:
#MAX_POSTS = 500

all_post_ids = []

HEADERS = {
    "User-Agent": "mia-natali-research/0.1",
    "Accept": "application/json,text/plain,*/*",
    "Accept-Language": "en-US,en;q=0.9",
}

subreddit = "nova"
search_url = f"https://www.reddit.com/r/{subreddit}/search.json"

params = {
    "q": 'datacenter',
    "restrict_sr": 1,
}

after = None
matched_posts = 0

with httpx.Client(headers=HEADERS, follow_redirects=True) as client:
    while True:
        if after is not None:
            params["after"] = after
        else:
            params.pop("after", None)

        r = client.get(search_url, params=params)
        print("REQUEST URL:", str(r.url))
        r.raise_for_status()
        listing = r.json()

        children = listing["data"]["children"]
        if not children:
            break

        for child in children:
            # data = child.get("data", {})
            # post_id = data.get("id")

            # title = (data.get("title") or "")
            # selftext = (data.get("selftext") or "")
            # text = (title + "\n" + selftext).lower()

            post_id = child["data"]["id"]
            all_post_ids.append(post_id)
            title = (child["data"]["title"] or "")
            selftext = (child["data"]["selftext"] or "")
            text = (title + " " + selftext).lower()

            
            
        after = listing["data"]["after"]
        print("After:", after)
        
        if after is None:
            break

        # if len(all_post_ids) >= MAX_POSTS:
        #     break

        time.sleep(0.2)

print("Total posts scanned:", len(all_post_ids))
print(all_post_ids)

REQUEST URL: https://www.reddit.com/r/nova/search.json?q=datacenter&restrict_sr=1
After: t3_sp9ms4
REQUEST URL: https://www.reddit.com/r/nova/search.json?q=datacenter&restrict_sr=1&after=t3_sp9ms4
After: t3_yv9you
REQUEST URL: https://www.reddit.com/r/nova/search.json?q=datacenter&restrict_sr=1&after=t3_yv9you
After: None
Total posts scanned: 55
['15cwyr6', 'xae8nw', '1sej8n4', '1rsspp8', '1lko81m', 'wktuak', '1p21o4w', '1244196', '1pezsl7', '1s7gqhr', 'eddta7', '1lg6mkl', '7htslh', '1pc87qw', 'w9hdeg', '10s4xtb', '1jffy5n', 'rleg0c', '1p4yg05', '1isnvbc', '1abhhob', '1po4dfm', '1kylm7v', '1ki7msg', 'sp9ms4', '1gawpld', 'esy3zu', '1r77l21', '1ndc2cd', '1kq8wz2', 'u7i038', '1ljjzo8', '1lqcva0', 'p9y987', '152m2sz', 'jrysr5', '1dxrg5p', '1edvmw5', '1al34nd', '9gtiez', '1fdrsh8', '117zom7', '2rv3ls', '1bdp55a', 'y1p6x4', '158wywd', '14gh0c1', '171nn12', '1341j3h', 'yv9you', 'k2bayk', '67nayl', '7i24g1', 'j25zf4', '8h0ztg']


In [19]:

posts = pd.DataFrame(columns=["ids", "text", "date", "votes", "number of comments"])
datacenters_keywords = ["datacenter", "data center", "datacentre", "data centre", "PW Digital Gateway", "Vantage", "Quantum Loophole"]
post_ids = []
post_texts = []
post_dates = []
post_votes = []
post_comment_numbers = []


for pid in all_post_ids:
    with httpx.Client(headers=HEADERS, follow_redirects=True) as client:
        post_url = f"https://www.reddit.com/comments/{pid}.json"                
        r = client.get(post_url)
        r.raise_for_status()
        post_json = r.json()
        text_and_title = post_json[0]["data"]["children"][0]["data"]["title"] + " " + post_json[0]["data"]["children"][0]["data"]["selftext"]
        text_and_title = text_and_title.replace('\n', ' ')
        if any(kw in text_and_title.lower() for kw in datacenters_keywords):
            for comment in post_json[1]["data"]["children"]:
                if comment.get("kind") != "t1":
                    continue
                if comment["data"]["score"] >= 10 or comment["data"]["score"] <= -10:
                    all_post_ids.append(comment["data"]["id"])
                    post_ids.append(comment["data"]["id"])
                    post_texts.append(comment["data"]["body"])
                    post_dates.append(comment["data"]["created_utc"])
                    post_votes.append(comment["data"]["score"])
                    post_comment_numbers.append(None)
        else:
            for comment in post_json[1]["data"]["children"]:
                if comment.get("kind") != "t1":
                    continue
                comment_text = comment["data"]["body"]
                if any(kw in comment_text.lower() for kw in datacenters_keywords):
                    text_and_title = text_and_title + " " + comment_text
        post_ids.append(pid)
        post_texts.append(text_and_title)
        post_dates.append(post_json[0]["data"]["children"][0]["data"]["created_utc"])
        post_votes.append(post_json[0]["data"]["children"][0]["data"]["score"])
        post_comment_numbers.append(post_json[0]["data"]["children"][0]["data"]["num_comments"])

print(len(post_ids))
print(len(posts["ids"]))
posts["ids"] = post_ids
posts["text"] = post_texts
posts["date"] = post_dates
posts["votes"] = post_votes
posts["number of comments"] = post_comment_numbers
print(len(all_post_ids))

ReadTimeout: The read operation timed out

In [ ]:
posts.to_csv("post_dataframe_1.csv", index=False)